In [ ]:
import pandas as pd
import numpy as np

transaction_base = pd.read_csv("/content/drive/MyDrive/transaction_base.csv")
fraud_base = pd.read_csv("/content/drive/MyDrive/fraud_base.csv")
customer_base = pd.read_csv("/content/drive/MyDrive/customer_base.csv")
card_base = pd.read_csv("/content/drive/MyDrive/card_base.csv")

In [ ]:
card_base.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   card_number   500 non-null    object
 1   card_family   500 non-null    object
 2   credit_limit  500 non-null    int64 
 3   cust_id       500 non-null    object
dtypes: int64(1), object(3)
memory usage: 15.8+ KB


In [ ]:
customer_base.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5674 entries, 0 to 5673
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   cust_id                 5674 non-null   object
 1   age                     5674 non-null   int64 
 2   customer_segment        5674 non-null   object
 3   customer_vintage_group  5674 non-null   object
dtypes: int64(1), object(3)
memory usage: 177.4+ KB


In [ ]:
fraud_base.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 109 entries, 0 to 108
Data columns (total 2 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   transaction_id  109 non-null    object
 1   fraud_flag      109 non-null    int64 
dtypes: int64(1), object(1)
memory usage: 1.8+ KB


In [ ]:
transaction_base.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   transaction_id       10000 non-null  object
 1   transaction_date     10000 non-null  object
 2   credit_card_id       10000 non-null  object
 3   transaction_value    10000 non-null  int64 
 4   transaction_segment  10000 non-null  object
dtypes: int64(1), object(4)
memory usage: 390.8+ KB


1. Using the credit card transaction data, calculate each customer's total transaction value across all their cards. Then find how many customers have a cumulative transaction value greater than 49,000.
Treat a customer as high-spend if the sum of all their transaction values across all cards exceeds 49,000.

In [ ]:
# Join transaction_base and card_base
df = transaction_base.merge(
    card_base,
    left_on='credit_card_id',
    right_on='card_number',
    how='inner'
)

# Aggregate total spend per customer
customer_spend = (
    df.groupby('cust_id', as_index=False)['transaction_value']
      .sum()
      .rename(columns={'transaction_value': 'total_spend'})
)

# Count customers with total_spend > 49000
result = customer_spend[customer_spend['total_spend'] > 49000]['cust_id'].nunique()

result


482

2. Define potential Premium card candidates as customers whose credit limit is in the top 10% of all credit limits
Using the card and customer data:
Calculate the 90th percentile of Credit_Limit
Identify all customers whose credit limit is greater than or equal to this threshold
Return their Cust_ID, Age, Customer_Segment, and Customer_Vintage_Group
You may assume that a customer with at least one card at or above this threshold is eligible for a Premium card.

In [ ]:
p90_credit_limit = card_base['credit_limit'].quantile(0.90)
required_credit = card_base.merge(customer_base, on = 'cust_id', how = 'inner')
required_credit = required_credit[required_credit['credit_limit'] > p90_credit_limit][['cust_id', 'age', 'customer_segment', 'customer_vintage_group']]
print(required_credit)

     cust_id  age customer_segment customer_vintage_group
5    CC66746   44          Diamond                    VG1
12   CC96267   24          Diamond                    VG1
20   CC43306   41         Platinum                    VG2
31   CC32209   47         Platinum                    VG2
32   CC99214   46         Platinum                    VG2
48   CC48867   30             Gold                    VG3
49   CC25727   31          Diamond                    VG1
92   CC23352   31          Diamond                    VG1
106  CC46521   44          Diamond                    VG1
132  CC45677   37         Platinum                    VG2
142  CC82682   20         Platinum                    VG2
151  CC51058   45         Platinum                    VG2
152  CC86871   30         Platinum                    VG2
154  CC93123   35          Diamond                    VG1
159  CC89243   42          Diamond                    VG1
168  CC88046   41          Diamond                    VG1
180  CC37292  

3. Using the fraud, transaction, and card tables:
Identify all cards that have at least one fraudulent transaction
From those cards, find the minimum and maximum credit limit
Return the overall min_credit_limit and max_credit_limit among cards involved in fraud.

In [ ]:
fraud_transaction = fraud_base.merge(transaction_base, on = 'transaction_id', how = 'inner')
fraud_card = fraud_transaction.merge(card_base, left_on = 'credit_card_id', right_on = 'card_number', how = 'inner')
result = pd.DataFrame({
    'min_credit_limit': [fraud_card['credit_limit'].min()],
    'max_credit_limit': [fraud_card['credit_limit'].max()]
})
result

,min_credit_limit,max_credit_limit
0,2000,879000


4. Using all four tables:
Identify customers who have at least one fraudulent transaction
Group them by Card_Family (card type)
For each card family, compute the average age of customers involved in fraud
Return one row per card family with its average age (rounded to 2 decimals).

In [ ]:
fraud_merge = fraud_base.merge(transaction_base, on = 'transaction_id',how = 'inner')
card_merge = fraud_merge.merge(card_base, left_on = 'credit_card_id', right_on = 'card_number', how = 'inner')
customer_merge = card_merge.merge(customer_base, on = 'cust_id', how = 'inner')
average_age = customer_merge.groupby('card_family', as_index=False)['age'].mean().round(2)
average_age

,card_family,age
0,Gold,36.62
1,Platinum,32.20
2,Premium,35.22


5. Using the fraud and transaction tables:
Consider only transactions flagged as fraud
Group them by calendar month
Find the month that has the highest count of fraudulent transactions
Return the fraud month (e.g. 2023-05-01 as month start) and the fraud count.

In [ ]:
fraud_merge = fraud_base.merge(transaction_base, on = 'transaction_id',how = 'inner')
fraud_merge['transaction_date'] = pd.to_datetime(fraud_merge['transaction_date'])
fraud_merge['fraud_month'] = fraud_merge['transaction_date'].dt.month
fraud_months = fraud_merge.groupby('fraud_month', as_index = False)['fraud_flag'].count().rename(columns={'fraud_flag':'fraud_count'})
fraud_months = fraud_months.sort_values(by='fraud_count', ascending=False).head(1)
fraud_months

,fraud_month,fraud_count
8,9,14


6. Using all relevant tables:
Consider only non-fraudulent transactions
For each customer, calculate the total transaction value of their non-fraud transactions
Exclude any customer who has at least one fraudulent transaction
Return the customer with the highest total non-fraud spend

In [ ]:
fraud_merge = fraud_base.merge(transaction_base, on = 'transaction_id',how = 'right')
card_merge = fraud_merge.merge(card_base, left_on = 'credit_card_id', right_on = 'card_number', how = 'inner')
card_merge['fraud_flag'] = card_merge['fraud_flag'].fillna(0)
non_fraud_transaction = card_merge.groupby('cust_id')[['transaction_value','fraud_flag']].sum()
non_fraud_transaction = non_fraud_transaction[non_fraud_transaction['fraud_flag'] == 0]['transaction_value'].reset_index()
non_fraud_transaction = non_fraud_transaction.sort_values(by = 'transaction_value', ascending = False).head(1)
non_fraud_transaction

,cust_id,transaction_value
340,CC91963,1448581


7. Using the customer, card, and transaction tables:
Identify customers who have never performed a single transaction on any of their cards
Return the list of such customer IDs. These are customers in Customer_base with zero matching transactions in Transaction_base.



In [ ]:
card_merge = transaction_base.merge(card_base, left_on = 'credit_card_id', right_on = 'card_number', how = 'inner')
non_transaction_customer = card_merge.merge(customer_base, on = 'cust_id', how = 'right')
non_transaction_customer = non_transaction_customer[non_transaction_customer['transaction_id'].isnull()]['cust_id']
non_transaction_customer

,cust_id
0,CC25034
1,CC59625
2,CC69314
3,CC67036
4,CC25597
...,...
15167,CC41159
15168,CC86108
15169,CC60493
15170,CC42924


8. Using the Card_base table, for each Card_Family:
Find the minimum credit limit
Find the maximum credit limit
Return one row per card family with its lowest and highest credit limit.

In [ ]:
result = card_base.groupby('card_family').agg(min_credit_limit = ('credit_limit', 'min'), max_credit_limit = ('credit_limit', 'max'))
result

,min_credit_limit,max_credit_limit
card_family,,
Gold,2000,50000
Platinum,51000,200000
Premium,108000,899000


9. Using customer, card, and transaction data:
Assign each customer to one of the above age brackets based on their Age
For each age bracket, calculate the total transaction value performed by customers in that bracket
Return one row per age bracket with the total transaction value.

In [ ]:
card_merge = transaction_base.merge(card_base, left_on = 'credit_card_id', right_on = 'card_number', how = 'inner')
customer_merge = card_merge.merge(customer_base, on = 'cust_id', how = 'inner')

customer_merge['age_bucket'] = np.select(
    [
        customer_merge['age'] < 20,
        customer_merge['age'].between(20, 29),
        customer_merge['age'].between(30, 39),
        customer_merge['age'].between(40, 49)
    ],
    [
        '0-20',
        '20-30',
        '30-40',
        '40-50'
    ],
    default='50+'
)

result = customer_merge.groupby('age_bucket', as_index=False)['transaction_value'].sum().rename(columns={'transaction_value': 'total_transaction_value'})

result

,age_bucket,total_transaction_value
0,20-30,71726649
1,30-40,81946625
2,40-50,87604704
3,50+,6309435


10. Using the transactions, cards, and fraud tables:
Consider only non-fraudulent transactions (transactions with no fraud record, or Fraud_Flag = 0)
Group by Card_Family
For each card family, compute:
Total number of non-fraud transactions
Total non-fraud transaction value
Identify the card family that has the highest number of non-fraud transactions and, in case of a tie, the highest total non-fraud transaction value.

In [ ]:
fraud_merge = fraud_base.merge(transaction_base, on = 'transaction_id',how = 'right')
card_merge = fraud_merge.merge(card_base, left_on = 'credit_card_id', right_on = 'card_number', how = 'inner')
card_merge['fraud_flag'] = card_merge['fraud_flag'].fillna(0)
non_fraud_transaction = card_merge[card_merge['fraud_flag'] == 0]
non_fraud_transaction = non_fraud_transaction.groupby('card_family', as_index=False).agg(non_fraud_txn_count=('fraud_flag', 'count'), non_fraud_total_value=('transaction_value', 'sum'))
result = non_fraud_transaction.sort_values(by=['non_fraud_txn_count', 'non_fraud_total_value'], ascending=[False, False]).head(1)
result

,card_family,non_fraud_txn_count,non_fraud_total_value
2,Premium,4054,100002750
